Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Google BigQuery Connector Samples

Read Google BigQuery tables and views through the Oracle AI Data Platform (AIDP) `GOOGLE_BIGQUERY` connector using the Progress DataDirect driver. The connector supports read-only ingestion and external-catalog access. Write operations are not supported.

## Prerequisites

- Use an AIDP notebook attached to compute with the Google BigQuery connector available (AIDP 5.1 or later).
- Configure a Google service account with permission to read the selected project, dataset and table and run the required BigQuery queries. Compute must be able to reach Google BigQuery and the Google OAuth token endpoint.
- Store the **full Google service-account JSON document** in an AIDP secret. Use `oauth.type=service_account` and pass the JSON as `oauth.pvt.key`; a PEM private key alone or a key-file path is not sufficient.
- Replace the placeholders below with your project, dataset, table, and AIDP secret identifiers. Run the direct-read sections in order. Catalog sections use an existing Google BigQuery external catalog created in **Master Catalogs**.

Keep credential values out of notebook outputs and source control.

## Configure a direct connection

Retrieve the service-account JSON from an AIDP secret. The email must match the service account in that JSON. The endpoint below is the standard BigQuery API endpoint.

In [ ]:
project_id = "<GCP_PROJECT_ID>"
dataset = "<DATASET>"
table = "<TABLE_OR_VIEW>"
service_account_email = "<SERVICE_ACCOUNT_EMAIL>"

service_account_json = aidputils.secrets.get("<SECRET_NAME>", "<SECRET_KEY>")

connection_options = {
    "type": "GOOGLE_BIGQUERY",
    "host": "https://www.googleapis.com/bigquery/v2",
    "port": "443",
    "project.id": project_id,
    "oauth.type": "service_account",
    "oauth.service.acct.email": service_account_email,
    "oauth.pvt.key": service_account_json,
}

## Read a table or view

BigQuery datasets map to the connector's `schema` option. `row.limit` bounds this sample's result size.

In [ ]:
bigquery_df = (
    spark.read.format("aidataplatform")
    .options(**connection_options)
    .option("schema", dataset)
    .option("table", table)
    .option("row.limit", "10")
    .load()
)
bigquery_df.show()

## Push down a projection and filter

Replace the column names and predicate with columns and a condition from your table. These options reduce the data returned by BigQuery.

In [ ]:
filtered_df = (
    spark.read.format("aidataplatform")
    .options(**connection_options)
    .option("schema", dataset)
    .option("table", table)
    .option("pushdown.columns", "<COLUMN_1>,<COLUMN_2>")
    .option("pushdown.filter", "<COLUMN_1> IS NOT NULL")
    .option("row.limit", "10")
    .load()
)
filtered_df.show()

## Read using BigQuery SQL

Use `pushdown.sql` for a source SQL query. Replace the placeholders in the query with a fully qualified BigQuery table or view. The backticks are BigQuery identifier quoting.

In [ ]:
query_df = (
    spark.read.format("aidataplatform")
    .options(**connection_options)
    .option("schema", dataset)
    .option("pushdown.sql", "SELECT * FROM `<GCP_PROJECT_ID>.<DATASET>.<TABLE_OR_VIEW>` LIMIT 10")
    .load()
)
query_df.show()

## Partition a numeric range read

Choose a numeric column and bounds appropriate for your data. Replace the illustrative `0` and `10000` bounds. Use the filter to select the intended range explicitly; partition bounds configure how the read is divided.

In [ ]:
partition_column = "<NUMERIC_COLUMN>"
partition_lower = "0"
partition_upper = "10000"

partitioned_df = (
    spark.read.format("aidataplatform")
    .options(**connection_options)
    .option("schema", dataset)
    .option("table", table)
    .option("partition.column", partition_column)
    .option("partition.num", "4")
    .option("partition.lower", partition_lower)
    .option("partition.upper", partition_upper)
    .option("pushdown.filter", f"{partition_column} >= {partition_lower} AND {partition_column} < {partition_upper}")
    .option("row.limit", "10")
    .load()
)
partitioned_df.show()

## Reuse an external catalog connection

Create a Google BigQuery external catalog in **Master Catalogs** with service-account authentication first. Use its catalog identifier as `catalog.id` to reuse the stored connection and credentials. Inline connection options are not needed for this read.

In [ ]:
catalog_df = (
    spark.read.format("aidataplatform")
    .option("catalog.id", "<CATALOG_ID>")
    .option("schema", "<DATASET>")
    .option("table", "<TABLE_OR_VIEW>")
    .option("row.limit", "10")
    .load()
)
catalog_df.show()

## Read with a three-part catalog name

Use the external catalog's **name**, followed by the BigQuery dataset and table or view. Quote each identifier separately with Spark backticks.

In [ ]:
external_df = spark.table("`<CATALOG_NAME>`.`<DATASET>`.`<TABLE_OR_VIEW>`")
external_df.limit(10).show()

## Query the external catalog with Spark SQL

Use the AIDP catalog name, dataset and table or view. Spark resolves the stored GBQ connection.


In [ ]:
external_sql_df = spark.sql(
    "SELECT * FROM `<CATALOG_NAME>`.`<DATASET>`.`<TABLE_OR_VIEW>` LIMIT 10"
)
external_sql_df.show()

## Connector options

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| type | GOOGLE_BIGQUERY | Y for inline connector config | Data source type. |
| catalog.id | Existing external catalog identifier | N | Reuse connection details from an existing external catalog instead of passing them inline. |
| host | BigQuery REST API endpoint | Y for inline connector config | For example, https://www.googleapis.com/bigquery/v2. |
| port | 443 | N (defaults to 443) | HTTPS port for the BigQuery endpoint. |
| project.id | GCP project ID | Y for inline connector config | GCP project that contains the target dataset. |
| oauth.type | service_account | Y for inline connector config | Authentication type for Progress BigQuery connectivity. |
| oauth.service.acct.email | Service-account email address | Y for inline connector config | Google service-account email. |
| oauth.pvt.key | Full service-account JSON document | Y for inline connector config | Obtain the JSON from an AIDP secret and pass it directly; do not use a PEM string, a key file path, or a connection.properties prefix. |
| schema | BigQuery dataset | Y | Dataset containing the table or view. |
| table | BigQuery table or view name | Y for table-based access | Target table or view name. |
| row.limit | Positive integer | N | Limits the number of rows read. |
| partition.column | Numeric column name | N | Column used for numeric range-based partitioned reads. |
| partition.num | Integer >= 1 | N | Number of partitions to use for range partitioning. |
| partition.lower | Numeric value | N | Lower bound for range partitioning. |
| partition.upper | Numeric value | N | Upper bound for range partitioning. |
| pushdown.columns | Comma-separated column names | N | Restricts the ingestion-style read to the requested columns. |
| pushdown.filter | SQL predicate | N | Predicate pushed down to BigQuery for ingestion-style reads. |
| pushdown.sql | BigQuery SQL query | N | Full SQL query for an ingestion-style read. |